# Exploratory Data Analysis — RACE Dataset\n## Intelligent Reading Comprehension & Quiz Generation System\n\nThis notebook explores the RACE (ReAding Comprehension from Examinations) dataset:\n- **28,000+ passages** and **~100,000 questions** from Chinese middle/high school English exams\n- Columns: `id`, `article`, `question`, `A`, `B`, `C`, `D`, `answer`\n- Splits: train (87,866), val (4,887), test (4,934)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

## 1. Load Dataset

In [ ]:
train_df = pd.read_csv('../data/raw/train.csv')
val_df   = pd.read_csv('../data/raw/val.csv')
test_df  = pd.read_csv('../data/raw/test.csv')

print(f"Train: {train_df.shape}")
print(f"Val:   {val_df.shape}")
print(f"Test:  {test_df.shape}")
print(f"\nColumns: {list(train_df.columns)}")
print(f"\nDtypes:\n{train_df.dtypes}")
print(f"\nMissing values:\n{train_df.isnull().sum()}")
train_df.head(3)

## 2. Summary Statistics Table

In [ ]:
# Summary statistics for all splits
summary_data = []
for name, df in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    summary_data.append({
        'Split': name,
        'Total Rows': len(df),
        'Unique Articles': df['article'].nunique(),
        'Avg Article Length (chars)': int(df['article'].str.len().mean()),
        'Avg Article Length (words)': int(df['article'].str.split().str.len().mean()),
        'Avg Question Length (words)': round(df['question'].str.split().str.len().mean(), 1),
        'Avg Option Length (words)': round(
            pd.concat([df['A'].str.split().str.len(), df['B'].str.split().str.len(),
                       df['C'].str.split().str.len(), df['D'].str.split().str.len()]).mean(), 1),
        'Missing Values': df.isnull().sum().sum()
    })

summary_df = pd.DataFrame(summary_data)
summary_df

## 3. Answer Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (name, df) in enumerate([('Train', train_df), ('Validation', val_df), ('Test', test_df)]):
    counts = df['answer'].value_counts().sort_index()
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
    axes[idx].bar(counts.index, counts.values, color=colors, edgecolor='black', linewidth=0.5)
    axes[idx].set_title(f'{name} Set — Answer Distribution', fontweight='bold')
    axes[idx].set_xlabel('Answer Label')
    axes[idx].set_ylabel('Count')
    for i, v in enumerate(counts.values):
        axes[idx].text(i, v + 100, f'{v}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../data/processed/answer_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("Answer distribution (Train):")
print(train_df['answer'].value_counts().sort_index())
print(f"\nClass balance ratio (max/min): {train_df['answer'].value_counts().max() / train_df['answer'].value_counts().min():.2f}")

## 4. Passage (Article) Length Analysis

In [ ]:
train_df['article_word_count'] = train_df['article'].str.split().str.len()
train_df['article_char_count'] = train_df['article'].str.len()
train_df['article_sentence_count'] = train_df['article'].apply(lambda x: len(re.split(r'[.!?]+', str(x))))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(train_df['article_word_count'], bins=50, color='#45B7D1', edgecolor='black', linewidth=0.5)
axes[0].set_title('Article Word Count Distribution', fontweight='bold')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].axvline(train_df['article_word_count'].mean(), color='red', linestyle='--', label=f"Mean: {train_df['article_word_count'].mean():.0f}")
axes[0].legend()

axes[1].hist(train_df['article_char_count'], bins=50, color='#96CEB4', edgecolor='black', linewidth=0.5)
axes[1].set_title('Article Character Count Distribution', fontweight='bold')
axes[1].set_xlabel('Character Count')
axes[1].set_ylabel('Frequency')
axes[1].axvline(train_df['article_char_count'].mean(), color='red', linestyle='--', label=f"Mean: {train_df['article_char_count'].mean():.0f}")
axes[1].legend()

axes[2].hist(train_df['article_sentence_count'], bins=30, color='#FF6B6B', edgecolor='black', linewidth=0.5)
axes[2].set_title('Article Sentence Count Distribution', fontweight='bold')
axes[2].set_xlabel('Sentence Count')
axes[2].set_ylabel('Frequency')
axes[2].axvline(train_df['article_sentence_count'].mean(), color='red', linestyle='--', label=f"Mean: {train_df['article_sentence_count'].mean():.0f}")
axes[2].legend()

plt.tight_layout()
plt.savefig('../data/processed/article_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("Article length statistics (words):")
print(train_df['article_word_count'].describe().to_string())

## 5. Question Length Analysis

In [ ]:
train_df['question_word_count'] = train_df['question'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train_df['question_word_count'], bins=40, color='#4ECDC4', edgecolor='black', linewidth=0.5)
axes[0].set_title('Question Word Count Distribution', fontweight='bold')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].axvline(train_df['question_word_count'].mean(), color='red', linestyle='--',
                label=f"Mean: {train_df['question_word_count'].mean():.1f}")
axes[0].legend()

# Option lengths
option_lengths = pd.DataFrame({
    'A': train_df['A'].str.split().str.len(),
    'B': train_df['B'].str.split().str.len(),
    'C': train_df['C'].str.split().str.len(),
    'D': train_df['D'].str.split().str.len()
})
axes[1].boxplot([option_lengths['A'].dropna(), option_lengths['B'].dropna(),
                 option_lengths['C'].dropna(), option_lengths['D'].dropna()],
                labels=['A', 'B', 'C', 'D'],
                patch_artist=True,
                boxprops=dict(facecolor='#FFD93D'))
axes[1].set_title('Answer Option Length Distribution (words)', fontweight='bold')
axes[1].set_xlabel('Option')
axes[1].set_ylabel('Word Count')

plt.tight_layout()
plt.savefig('../data/processed/question_option_lengths.png', dpi=150, bbox_inches='tight')
plt.show()

print("Question length statistics (words):")
print(train_df['question_word_count'].describe().to_string())

## 6. Question Type Analysis (Wh-words)

In [ ]:
wh_words = ['what', 'which', 'who', 'where', 'when', 'why', 'how']

def classify_question_type(question):
    q_lower = str(question).lower().strip()
    for wh in wh_words:
        if q_lower.startswith(wh):
            return wh.capitalize()
    if '______' in q_lower or '___' in q_lower or '_' in q_lower:
        return 'Fill-in-blank'
    return 'Other'

train_df['question_type'] = train_df['question'].apply(classify_question_type)

type_counts = train_df['question_type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = plt.cm.Set3(np.linspace(0, 1, len(type_counts)))
axes[0].barh(type_counts.index[::-1], type_counts.values[::-1], color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title('Question Types by Frequency', fontweight='bold')
axes[0].set_xlabel('Count')
for i, v in enumerate(type_counts.values[::-1]):
    axes[0].text(v + 200, i, f'{v} ({v/len(train_df)*100:.1f}%)', va='center', fontsize=9)

axes[1].pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 9})
axes[1].set_title('Question Type Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/processed/question_types.png', dpi=150, bbox_inches='tight')
plt.show()

print("Question type counts:")
print(type_counts.to_string())

## 7. Difficulty Level Analysis (Middle vs High School)

In [ ]:
# Difficulty can be inferred from the ID prefix: 'high' vs 'middle'
train_df['difficulty'] = train_df['id'].apply(
    lambda x: 'High School' if str(x).startswith('high') else 'Middle School'
)

diff_counts = train_df['difficulty'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(diff_counts.index, diff_counts.values,
            color=['#FF6B6B', '#4ECDC4'], edgecolor='black', linewidth=0.5)
axes[0].set_title('Difficulty Level Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(diff_counts.values):
    axes[0].text(i, v + 500, f'{v}\n({v/len(train_df)*100:.1f}%)', ha='center', fontsize=10)

# Compare article lengths across difficulty
train_df.boxplot(column='article_word_count', by='difficulty', ax=axes[1],
                 patch_artist=True,
                 boxprops=dict(facecolor='#FFD93D'),
                 medianprops=dict(color='red'))
axes[1].set_title('Article Length by Difficulty', fontweight='bold')
axes[1].set_xlabel('Difficulty')
axes[1].set_ylabel('Word Count')
plt.suptitle('')

plt.tight_layout()
plt.savefig('../data/processed/difficulty_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("Difficulty level statistics:")
print(train_df.groupby('difficulty')['article_word_count'].describe().to_string())

## 8. Top Words Analysis (Vocabulary Exploration)

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Get word frequencies from articles (excluding stopwords)
all_words = []
for text in train_df['article'].dropna().sample(10000, random_state=42):
    words = re.findall(r'\b[a-z]+\b', text.lower())
    all_words.extend([w for w in words if w not in ENGLISH_STOP_WORDS and len(w) > 2])

word_freq = Counter(all_words).most_common(30)
words, freqs = zip(*word_freq)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(list(words)[::-1], list(freqs)[::-1], color='#45B7D1', edgecolor='black', linewidth=0.5)
axes[0].set_title('Top 30 Words in Articles (excl. stopwords)', fontweight='bold')
axes[0].set_xlabel('Frequency (10K sample)')

# Question word frequencies
q_words = []
for text in train_df['question'].dropna():
    qw = re.findall(r'\b[a-z]+\b', text.lower())
    q_words.extend([w for w in qw if w not in ENGLISH_STOP_WORDS and len(w) > 2])

q_freq = Counter(q_words).most_common(20)
qw, qf = zip(*q_freq)

axes[1].barh(list(qw)[::-1], list(qf)[::-1], color='#FF6B6B', edgecolor='black', linewidth=0.5)
axes[1].set_title('Top 20 Words in Questions (excl. stopwords)', fontweight='bold')
axes[1].set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('../data/processed/word_frequencies.png', dpi=150, bbox_inches='tight')
plt.show()

total_vocab = set()
for text in train_df['article'].dropna().sample(10000, random_state=42):
    total_vocab.update(re.findall(r'\b[a-z]+\b', text.lower()))
print(f"Vocabulary size (10K article sample): {len(total_vocab):,}")

## 9. Correct Answer Position vs Option Length

In [ ]:
# Check if correct answer tends to be longer/shorter
train_df['correct_option_len'] = train_df.apply(
    lambda row: len(str(row[row['answer']]).split()) if row['answer'] in ['A','B','C','D'] else 0, axis=1
)

all_option_lens = []
for _, row in train_df.iterrows():
    for opt in ['A', 'B', 'C', 'D']:
        is_correct = (opt == row['answer'])
        all_option_lens.append({
            'option': opt,
            'word_count': len(str(row[opt]).split()),
            'is_correct': 'Correct' if is_correct else 'Incorrect'
        })
    if len(all_option_lens) > 40000:  # Sample for speed
        break

opt_df = pd.DataFrame(all_option_lens)

fig, ax = plt.subplots(figsize=(10, 5))
opt_df.boxplot(column='word_count', by='is_correct', ax=ax,
               patch_artist=True,
               boxprops=dict(facecolor='#96CEB4'),
               medianprops=dict(color='red'))
ax.set_title('Option Word Count: Correct vs Incorrect', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Word Count')
plt.suptitle('')
plt.tight_layout()
plt.savefig('../data/processed/correct_vs_incorrect_length.png', dpi=150, bbox_inches='tight')
plt.show()

print("Correct answer option length stats:")
print(f"  Mean: {opt_df[opt_df['is_correct']=='Correct']['word_count'].mean():.2f}")
print(f"Incorrect option length stats:")
print(f"  Mean: {opt_df[opt_df['is_correct']=='Incorrect']['word_count'].mean():.2f}")

## 10. Sample Data Inspection

In [ ]:
# Display 3 random samples in a readable format
samples = train_df.sample(3, random_state=42)
for idx, row in samples.iterrows():
    print("=" * 80)
    print(f"ID: {row['id']} | Correct Answer: {row['answer']} | Difficulty: {row['difficulty']}")
    print("-" * 80)
    print(f"ARTICLE (first 300 chars): {row['article'][:300]}...")
    print(f"\nQUESTION: {row['question']}")
    print(f"  A: {row['A']}")
    print(f"  B: {row['B']}")
    print(f"  C: {row['C']}")
    print(f"  D: {row['D']}")
    print(f"  CORRECT: {row['answer']} -> {row[row['answer']]}")
    print()

## 11. Key EDA Findings\n\n**Summary of observations:**\n1. **Dataset size**: ~87K training samples with 8 columns each\n2. **Answer balance**: Slight class imbalance — C is most frequent, A is least\n3. **Article lengths**: Vary significantly (middle school shorter, high school longer)\n4. **Question types**: Mix of Wh-questions and fill-in-the-blank\n5. **Difficulty split**: Dataset contains both middle and high school level questions\n6. **Option lengths**: Correct and incorrect options have similar word counts (no easy shortcut)\n7. **No missing values**: Dataset is clean and ready for preprocessing